# Nepali Wikipedia Scraper

This notebook scrapes Nepali-language content from Wikipedia (`ne.wikipedia.org`) for a list of article titles supplied in a text file (one title per line), and saves each article's text into its own `.txt` file.

**Workflow:**
1. Install/import dependencies
2. Upload your titles file (`titles.txt`)
3. Configure output settings
4. Run the scraper — each article is saved as a separate file in `nepali_wiki_articles/`
5. Download the results as a zip

**Titles file format:** plain text, one Wikipedia article title per line, in Nepali (Devanagari) script, e.g.:
```
काठमाडौं
पोखरा
सगरमाथा
```

## 1. Install and import dependencies

In [ ]:
!pip install -q wikipedia-api requests

In [ ]:
import os
import re
import time
import unicodedata
import requests
import wikipediaapi
from pathlib import Path

## 2. Upload your titles file

Run the cell below and choose your `titles.txt` file (one Nepali article title per line). If you're not running this in Colab, place the file in the working directory and set `TITLES_FILE` accordingly in the next section.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    TITLES_FILE = list(uploaded.keys())[0]
    print(f"Uploaded file: {TITLES_FILE}")
except ImportError:
    # Not running in Colab -- set the path manually
    TITLES_FILE = "titles.txt"
    print("Not running in Colab. Set TITLES_FILE manually to point to your titles file.")

## 3. Configuration

- `OUTPUT_DIR`: folder where individual article text files will be saved
- `LANG`: Wikipedia language code (`ne` = Nepali)
- `SLEEP_SECONDS`: delay between requests to be polite to Wikipedia's servers
- `INCLUDE_SECTIONS`: whether to keep section headings in the saved text

In [ ]:
OUTPUT_DIR = "nepali_wiki_articles"
LANG = "ne"
SLEEP_SECONDS = 1.0
INCLUDE_SECTIONS = True
USER_AGENT = "NepaliKGResearchScraper/1.0 (research use; contact: your_email@example.com)"

os.makedirs(OUTPUT_DIR, exist_ok=True)

wiki = wikipediaapi.Wikipedia(
    user_agent=USER_AGENT,
    language=LANG,
    extract_format=wikipediaapi.ExtractFormat.WIKI,
)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Wikipedia language: {LANG} ({wiki.language})")

## 4. Helper functions

In [ ]:
def read_titles(path):
    """Read titles from a text file, one per line. Strips blank lines and whitespace."""
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f]
    return [t for t in lines if t]


def safe_filename(title, max_len=150):
    """Convert an article title (Devanagari or otherwise) into a safe filename."""
    normalized = unicodedata.normalize("NFC", title)
    # Replace characters that are unsafe in filenames
    cleaned = re.sub(r'[\\/*?:"<>|]', "", normalized)
    cleaned = cleaned.strip().replace(" ", "_")
    if not cleaned:
        cleaned = "untitled"
    return cleaned[:max_len]


def extract_article_text(page, include_sections=True):
    """Build the full text of an article, optionally including section headings."""
    if not include_sections:
        return page.text

    lines = [page.title, "=" * len(page.title), ""]
    lines.append(page.summary)
    lines.append("")

    def walk_sections(sections, level=2):
        for section in sections:
            lines.append("#" * level + " " + section.title)
            lines.append("")
            if section.text:
                lines.append(section.text)
                lines.append("")
            walk_sections(section.sections, level=min(level + 1, 6))

    walk_sections(page.sections)
    return "\n".join(lines)


def scrape_titles(titles, output_dir, lang_wiki, sleep_seconds=1.0, include_sections=True):
    """Fetch each title from Wikipedia and save its content to a separate file.

    Returns a summary dict of successes, redirects, and failures.
    """
    results = {"success": [], "not_found": [], "error": []}
    used_filenames = set()

    for i, title in enumerate(titles, 1):
        print(f"[{i}/{len(titles)}] Fetching: {title}")
        try:
            page = lang_wiki.page(title)

            if not page.exists():
                print(f"  -> NOT FOUND: {title}")
                results["not_found"].append(title)
                continue

            actual_title = page.title  # resolves redirects to canonical title
            fname = safe_filename(actual_title)

            # Avoid overwriting files when titles collide after sanitizing / redirect resolution
            candidate = fname
            suffix = 1
            while candidate in used_filenames:
                suffix += 1
                candidate = f"{fname}_{suffix}"
            used_filenames.add(candidate)

            filepath = os.path.join(output_dir, f"{candidate}.txt")
            content = extract_article_text(page, include_sections=include_sections)

            with open(filepath, "w", encoding="utf-8") as f:
                f.write(content)

            print(f"  -> Saved to {filepath} ({len(content)} chars)")
            results["success"].append({
                "requested_title": title,
                "resolved_title": actual_title,
                "file": filepath,
            })

        except Exception as e:
            print(f"  -> ERROR: {e}")
            results["error"].append({"title": title, "error": str(e)})

        time.sleep(sleep_seconds)

    return results

## 5. Run the scraper

In [ ]:
titles = read_titles(TITLES_FILE)
print(f"Loaded {len(titles)} titles from {TITLES_FILE}")
print(titles[:10])

summary = scrape_titles(
    titles,
    output_dir=OUTPUT_DIR,
    lang_wiki=wiki,
    sleep_seconds=SLEEP_SECONDS,
    include_sections=INCLUDE_SECTIONS,
)

print("\n--- Summary ---")
print(f"Success: {len(summary['success'])}")
print(f"Not found: {len(summary['not_found'])}")
print(f"Errors: {len(summary['error'])}")

if summary["not_found"]:
    print("\nTitles not found:")
    for t in summary["not_found"]:
        print(f"  - {t}")

if summary["error"]:
    print("\nTitles with errors:")
    for e in summary["error"]:
        print(f"  - {e['title']}: {e['error']}")

### (Optional) Retry not-found titles via Wikipedia search

Some titles may fail an exact match due to slightly different spelling or capitalization. This cell uses the Wikipedia search API to find the closest matching article title and retries scraping for those.

In [ ]:
def search_best_match(title, lang="ne"):
    """Query the Wikipedia search API for the closest matching title."""
    url = f"https://{lang}.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "search",
        "srsearch": title,
        "format": "json",
        "srlimit": 1,
    }
    headers = {"User-Agent": USER_AGENT}
    resp = requests.get(url, params=params, headers=headers, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    hits = data.get("query", {}).get("search", [])
    return hits[0]["title"] if hits else None


if summary["not_found"]:
    retry_titles = []
    for t in summary["not_found"]:
        match = search_best_match(t, lang=LANG)
        if match:
            print(f"'{t}' -> best match: '{match}'")
            retry_titles.append(match)
        else:
            print(f"'{t}' -> no match found")
        time.sleep(SLEEP_SECONDS)

    if retry_titles:
        retry_summary = scrape_titles(
            retry_titles,
            output_dir=OUTPUT_DIR,
            lang_wiki=wiki,
            sleep_seconds=SLEEP_SECONDS,
            include_sections=INCLUDE_SECTIONS,
        )
        print(f"\nRetry success: {len(retry_summary['success'])} / {len(retry_titles)}")
else:
    print("No not-found titles to retry.")

## 6. Download results as a zip

Bundles the `nepali_wiki_articles/` folder (all individually saved article files) into a zip and downloads it (Colab only).

In [ ]:
import shutil

zip_name = f"{OUTPUT_DIR}.zip"
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print(f"Created {zip_name}")

try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    print(f"Not running in Colab. Find your zip at: {os.path.abspath(zip_name)}")

## Notes

- Each article's text (summary + sections) is saved to its own `.txt` file, named after the (redirect-resolved) article title, inside `nepali_wiki_articles/`.
- The `wikipedia-api` library automatically follows redirects, so a title like `पोखरा उपमहानगरपालिका` resolving to a canonical page will be saved under the canonical title.
- Adjust `SLEEP_SECONDS` if you're scraping a very large title list, to stay within Wikipedia's usage etiquette.
- Set `INCLUDE_SECTIONS = False` in step 3 if you only want the plain flat article text without section markers.